# Análisis de datos: Extracción de datos desde un API REST 🌐📊

## Objetivos
- Entender qué es un **API** y por qué es una fuente común de datos reales.
- Conocer cómo funcionan las **llamadas HTTP**.
- Aprender la estructura básica de llamadas **GET** y **POST** usando `requests` en Python.
- Sentar las bases para usar APIs como fuente de datos en proyectos de análisis.

---

## Qué es un API
Un **API (Application Programming Interface)** es una interfaz que permite que dos sistemas se comuniquen entre sí.  
Desde el punto de vista de análisis de datos, un API es una **fuente externa de datos** a la que accedemos bajo reglas claras: qué pedir, cómo pedirlo y qué formato recibimos.

> 🧠 Piensa en un API como un *mesero*: tú haces el pedido (request), la cocina procesa la solicitud y te entrega exactamente lo que pediste (response).

Los APIs REST suelen devolver datos en formatos como **JSON**, ideales para ser consumidos por Python.

---

### Llamadas HTTP (GET, POST)
Las **llamadas HTTP** son la forma estándar de comunicarnos con un API. Cada llamada indica **qué queremos hacer**.

- **GET** 👉 Obtener información  
- **POST** 👉 Enviar información  
- (Existen otras como PUT, DELETE, pero nos enfocaremos en las más comunes)

> 📌 En análisis de datos, **GET es la más utilizada**, porque normalmente queremos *leer datos*, no modificarlos.

---

### Diferencia entre un API y una base de datos
Aunque ambos permiten acceder a datos, **no son lo mismo**:

- **Base de datos (SQL)**  
  - Acceso directo a tablas
  - Tú defines la consulta (`SELECT`, `JOIN`, `WHERE`)
  - Control total del esquema
  - Normalmente datos internos de una empresa

- **API**  
  - Acceso indirecto a datos
  - El proveedor define qué puedes pedir
  - No ves las tablas internas
  - Ideal para datos externos o en tiempo real

> 🔑 Un analista **no controla el API**, solo aprende a consumirlo correctamente.

---

### Llamadas a API en Python usando `requests`
En Python, la librería más común para trabajar con APIs es **`requests`**.  
Nos permite construir fácilmente llamadas HTTP y manejar las respuestas.

Una llamada a un API suele componerse de:
- **URL (endpoint)**
- **Parámetros (`params`)**
- **Headers**
- *(opcional)* Body (en POST)

Los **headers** son metadatos que acompañan la solicitud HTTP.  
Le dicen al servidor **quién eres**, **qué tipo de datos esperas** y **cómo debe procesarse la solicitud**.

Headers comunes en APIs:
- `Authorization` o `X-API-Key` → autenticación
- `Content-Type` → formato de datos enviados
- `Accept` → formato de datos esperados en la respuesta


---

#### Estructura de una llamada GET en `requests`
Una llamada **GET** se usa para **obtener datos** desde un API.

```python
import requests

url = "https://api.ejemplo.com/data"

params = {
    "limit": 100,
    "country": "MX"
}

response = requests.get(url, params=params)

response.status_code
```

Elementos clave:
- **url**: endpoint del API
- **params**: parámetros de consulta (query params)
- **response**: objeto con la respuesta del servidor

Para trabajar con los datos:
```python
data = response.json()
data
```

---

#### Estructura de una llamada POST en `requests`
Una llamada **POST** se usa para **enviar datos** al API (crear o procesar información).

```python
import requests

url = "https://api.ejemplo.com/data"

payload = {
    "name": "Roman",
    "role": "Data Analyst"
}

response = requests.post(url, json=payload)

response.status_code
```

Elementos clave:
- **json / data**: información que se envía al servidor
- **POST** modifica o crea recursos
- Menos común en análisis, más común en sistemas transaccionales

> ⚠️ En proyectos de análisis, normalmente **NO usamos POST**, pero es importante conocerlo para entender la lógica de los APIs.




## Ejemplo práctico: OpenAQ API – Acceso a datos ambientales 🌍🌫️

**OpenAQ** es una plataforma abierta que recopila y estandariza datos de **calidad del aire** provenientes de estaciones oficiales alrededor del mundo.  
Su API permite acceder a mediciones reales de contaminantes como **PM2.5, PM10, NO₂, O₃, CO**, entre otros, junto con información temporal y geográfica, lo que la convierte en una fuente ideal para análisis ambientales, series de tiempo y estudios comparativos entre ciudades o países.

> 🧠 OpenAQ no genera datos: **agrega y normaliza** información de múltiples proveedores oficiales para facilitar su análisis.

---

### Acceso a la API
Para consumir datos desde la **API v3 de OpenAQ** es necesario contar con un **API Key**, que se utiliza para autenticar cada solicitud.

El proceso es el siguiente:
1. Registrarse en la plataforma de OpenAQ Explorer.
2. Crear una cuenta de usuario.
3. Generar un **API Key** desde el panel de configuración.
4. Usar ese API Key en los **headers** de cada request.

> 🔐 El API Key funciona como una credencial personal.  
> No debe compartirse ni subirse a repositorios públicos.

En la práctica, el API Key se envía en el header `X-API-Key` y se recomienda almacenarlo como **variable de entorno**, especialmente cuando se trabaja con notebooks que luego se publicarán en GitHub.

Este paso habilita el acceso a endpoints como:
- Listado de países y ubicaciones
- Sensores de medición
- Mediciones crudas o agregadas (diarias)
- Series temporales ambientales reales

En las siguientes secciones utilizaremos este acceso para **extraer, transformar y analizar** datos ambientales reales usando Python. En especifico analizaremos el comportamiento de dos contaminantes $CO$ y $PM_{2.5}$


- **PM2.5 (material particulado fino)**: está compuesto por partículas microscópicas que pueden penetrar profundamente en los pulmones e incluso ingresar al torrente sanguíneo. Es uno de los contaminantes con mayor impacto en la salud, asociado a enfermedades respiratorias y cardiovasculares, por lo que se considera un **indicador crítico de calidad del aire** en estudios ambientales.

- **CO (monóxido de carbono)**: es un gas producto de la combustión incompleta, especialmente del tráfico vehicular. Aunque en concentraciones ambientales suele ser menos letal que otros contaminantes, es un **buen proxy de emisiones por combustión**, lo que lo hace muy relevante para analizar patrones urbanos y su relación con otros contaminante


In [ ]:
import requests
import pandas as pd
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
API_KEY=""

**Obtener el listado de paises**

In [ ]:


# Endpoint v3 Countries
url = "https://api.openaq.org/v3/countries"


headers = {
    "X-API-Key": API_KEY
}

params={'limit':1000}
response = requests.get(url, headers=headers,params=params)
response.raise_for_status()

data = response.json()

countries=pd.DataFrame(data['results'])

In [ ]:
countries

In [ ]:
countries.query('code == "MX"')

In [ ]:
# Endpoint v3 Countries
url = "https://api.openaq.org/v3/countries/157"


headers = {
    "X-API-Key": API_KEY
}

params={'limit':100000}
response = requests.get(url, headers=headers)
response.raise_for_status()

data = response.json()



**Obtener localidades de México**

In [ ]:
# Get locations

url = "https://api.openaq.org/v3/locations"


headers = {
    "X-API-Key": API_KEY
}

params={'limit':1000,'countries_id':[157]}
response = requests.get(url, headers=headers,params=params)
response.raise_for_status()

data = response.json()



In [ ]:
locations_sensors=pd.DataFrame(data['results'])
locations_sensors=locations_sensors.explode('sensors')
sensor_data=pd.json_normalize(locations_sensors['sensors']).reset_index(drop=True)
locations_sensors=locations_sensors[['id','name','locality']].reset_index(drop=True)
locations_sensors.rename(columns={'id':'id_location','name':'name_location'},inplace=True)
locations_sensors = pd.concat([locations_sensors,sensor_data],axis=1)
locations_sensors

In [ ]:
locations_sensors.query('locality == "GUANAJUATO" and name == "pm25 µg/m³"')

**Obtener mediciones de CO en Guanajuato para Enero 2025**

In [ ]:
ids_selection=locations_sensors.query('locality == "GUANAJUATO" and name == "co ppm"')['id'].to_list()

In [ ]:
ids_selection

In [ ]:
SENSOR_ID = 3761  # ← reemplaza por el sensor real

url = f"https://api.openaq.org/v3/sensors/{SENSOR_ID}/measurements/daily"

params = {
    "datetime_from": "2025-01-01T00:00:00Z",
    "datetime_to": "2025-01-31T23:59:59Z",
    "limit": 31
}

headers = {
    "X-API-Key": API_KEY
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

data = response.json()['results']

df = pd.json_normalize(data)
df.head()

In [ ]:
co_df=pd.DataFrame()
for sensor_id in ids_selection:
    print(f"Sensor {sensor_id}", end=' ')
    try:
        
        url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"

        params = {
            "datetime_from": "2025-01-01T00:00:00Z",
            "datetime_to": "2025-01-31T23:59:59Z",
            "limit": 31
        }

        headers = {
            "X-API-Key": API_KEY
        }

        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()

        data = response.json()['results']

        df = pd.json_normalize(data)
        df['sensor_id'] =sensor_id
        df['date_at']=pd.to_datetime(df['period.datetimeFrom.local']).dt.date
        df.rename(columns={'parameter.name':'parameter'},inplace=True)
        co_df=pd.concat([co_df,df[['date_at','sensor_id','parameter','value']]])
        time.sleep(0.5)
        print("Success")
    except Exception as ee:
        print("Failed")
        print (ee)
        pass

*Analisis exploratorio*

In [ ]:
co_df.info()

In [ ]:
co_df.describe()


In [ ]:
co_df['sensor_id']=co_df['sensor_id'].astype(str)
sns.boxplot(data=co_df,x='value',y='sensor_id')
plt.title('Variabilidad por sensor')

In [ ]:
co_df_daily=co_df.groupby('date_at')['value'].mean()

In [ ]:
co_df_daily

In [ ]:
co_df_daily.plot(figsize=(12,4), title="Concentración de CO (GT) en el tiempo")

**Obtener mediciones de PM25 en Guanajuato para Enero 2025**

In [ ]:
ids_selection=locations_sensors.query('locality == "GUANAJUATO" and name == "pm25 µg/m³"')['id'].to_list()

pm_df=pd.DataFrame()
for sensor_id in ids_selection:
    print(f"Sensor {sensor_id}", end=' ')
    try:
        
        url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"

        params = {
            "datetime_from": "2025-01-01T00:00:00Z",
            "datetime_to": "2025-01-31T23:59:59Z",
            "limit": 31
        }

        headers = {
            "X-API-Key": API_KEY
        }

        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()

        data = response.json()['results']

        df = pd.json_normalize(data)
        df['sensor_id'] =sensor_id
        df['date_at']=pd.to_datetime(df['period.datetimeFrom.local']).dt.date
        df.rename(columns={'parameter.name':'parameter'},inplace=True)
        pm_df=pd.concat([pm_df,df[['date_at','sensor_id','parameter','value']]])
        time.sleep(0.5)
        print("Success")
    except Exception as ee:
        print("Failed")
        print (ee)
        pass

In [ ]:
pm_df.head()

In [ ]:
pm_df.describe()

In [ ]:
pm_df['sensor_id']=pm_df['sensor_id'].astype(str)
sns.boxplot(data=pm_df,x='value',y='sensor_id')
plt.title('Variabilidad por sensor')

In [ ]:
pm_df_daily=pm_df.groupby('date_at')['value'].mean()

In [ ]:
pm_df_daily.plot(figsize=(12,4), title="Concentración de PM25 (GT) en el tiempo")

**Analisis de correlación**

In [ ]:
df_daily=pm_df_daily.reset_index(name='pm25').merge(co_df_daily.reset_index(name='co'))

In [ ]:
df_daily

In [ ]:
df_daily[["pm25", "co"]].corr()

In [ ]:
plt.figure(figsize=(6,5))

sns.scatterplot(
    data=df_daily,
    x="pm25",
    y="co"
)

plt.title("Relación entre PM2.5 y CO")
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("CO")
plt.grid(True)
plt.show()

### Conclusiones

*Escribe aqui tus conclusiones*


### Compartir resultados

**Instrucciones**

*Carga tu reporte en GitHub, incluye en el repositorio un archivo README donde describas el objetivo del estudio*

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨